# Практика 9. Перетворення даних: похідні змінні, дискретизація, масштабування, індикаторне кодування

**Варіант 1 — Київ.** Базова ціна: 850 грн; базова кількість: 3.

Мета: побудувати похідні змінні, розбити числову змінну на групи двома способами, масштабувати змінні різного порядку величини та закодувати номінальну категорію one-hot способом.

## Завдання 1. Побудова набору клієнтів

Кожен рядок — одне замовлення клієнта інтернет-магазину. Вік, дохід і кількість відповідають реалістичним значенням, а міста доставки охоплюють кілька варіантів із таблиці курсу.

In [1]:
import pandas as pd

variant = 1
city = 'Київ'
base_price = 850
base_quantity = 3

orders = pd.DataFrame({
    'вік': [18, 19, 20, 21, 22, 28, 35, 45, 58, 70],
    'дохід': [18000, 22000, 26000, 30000, 36000, 45000, 58000, 76000, 115000, 180000],
    'кількість': [2, 4, 1, 3, 5, 2, 6, 3, 4, 7],
    'місто_доставки': ['Київ', 'Львів', 'Одеса', 'Київ', 'Харків',
                       'Дніпро', 'Львів', 'Одеса', 'Київ', 'Харків']
})

print(f'Варіант {variant}: {city}; базова ціна: {base_price} грн; базова кількість: {base_quantity}')
display(orders)
assert len(orders) == 10
assert orders['місто_доставки'].nunique() >= 4

Варіант 1: Київ; базова ціна: 850 грн; базова кількість: 3


,вік,дохід,кількість,місто_доставки
0,18,18000,2,Київ
1,19,22000,4,Львів
2,20,26000,1,Одеса
3,21,30000,3,Київ
4,22,36000,5,Харків
5,28,45000,2,Дніпро
6,35,58000,6,Львів
7,45,76000,3,Одеса
8,58,115000,4,Київ
9,70,180000,7,Харків


## Завдання 2. Похідні змінні

`сума_замовлення` показує абсолютну вартість кошика, а `частка_доходу` нормує цю суму на дохід конкретного клієнта. Остання ознака дає змогу порівнювати фінансове навантаження замовлень для клієнтів із різним рівнем доходу.

In [ ]:
orders['сума_замовлення'] = base_price * orders['кількість']
orders['частка_доходу'] = orders['сума_замовлення'] / orders['дохід']

display(orders[['дохід', 'кількість', 'сума_замовлення', 'частка_доходу']])
assert (orders['сума_замовлення'] == base_price * orders['кількість']).all()
assert (orders['частка_доходу'] == orders['сума_замовлення'] / orders['дохід']).all()

## Завдання 3. Дискретизація віку

`pd.cut()` створює інтервали однакової ширини, а `pd.qcut()` формує групи за квантилями, щоб у кожній групі було приблизно однаково спостережень.

In [ ]:
orders['вікова_група_cut'] = pd.cut(orders['вік'], bins=3)
orders['вікова_група_qcut'] = pd.qcut(orders['вік'], q=3)

cut_counts = orders['вікова_група_cut'].value_counts().sort_index()
qcut_counts = orders['вікова_група_qcut'].value_counts().sort_index()
print('Кількість клієнтів у групах cut:')
display(cut_counts.to_frame('кількість'))
print('Кількість клієнтів у групах qcut:')
display(qcut_counts.to_frame('кількість'))
assert cut_counts.sum() == 10
assert qcut_counts.sum() == 10
assert cut_counts.tolist() != qcut_counts.tolist()

**Порівняння.** `cut` ділить весь числовий діапазон на три рівні за довжиною інтервали, тому через нерівномірний розподіл віку групи мають різні розміри. `qcut` використовує ранги й квантили, тому кількість клієнтів у групах близька до однакової, але самі вікові межі мають різну ширину.

## Завдання 4. Масштабування

До перетворення вік має десятки одиниць, а дохід — десятки й сотні тисяч гривень. Застосуємо min-max нормалізацію та z-оцінку окремо до кожної змінної.

In [ ]:
numeric = orders[['вік', 'дохід']]
min_max = (numeric - numeric.min()) / (numeric.max() - numeric.min())
z_score = (numeric - numeric.mean()) / numeric.std()

min_max.columns = ['вік_minmax', 'дохід_minmax']
z_score.columns = ['вік_z', 'дохід_z']
scaled_preview = pd.concat([min_max, z_score], axis=1)
display(scaled_preview.head())

assert min_max.min().min() == 0
assert min_max.max().max() == 1
assert abs(z_score.mean()).max() < 1e-12
assert abs(z_score.std() - 1).max() < 1e-12

**Пояснення.** Без масштабування дохід мав би набагато більший числовий внесок, ніж вік, навіть якщо обидві змінні однаково важливі за змістом. Це спотворило б відстані, ваги ознак і регуляризацію. Min-max переводить значення в `[0, 1]`, а z-оцінка — у шкалу зі середнім 0 і стандартним відхиленням 1.

## Завдання 5. Індикаторне кодування

Місто доставки є номінальною змінною: між Києвом, Львовом, Одесою та Харковом немає природного порядку. Тому використаємо `pd.get_dummies()`, а не мапінг у числа 1, 2, 3.

In [ ]:
city_dummies = pd.get_dummies(orders['місто_доставки'], prefix='місто')
orders_encoded = pd.concat([orders, city_dummies], axis=1)
display(city_dummies)
display(orders_encoded.head())

unique_cities = orders['місто_доставки'].nunique()
print('Унікальних міст:', unique_cities)
print('Нових індикаторних стовпців:', city_dummies.shape[1])
assert city_dummies.shape[1] == unique_cities

## Контрольні питання

1. **`cut` і `qcut`:** `cut` задає інтервали однакової довжини, тоді як `qcut` ділить відсортовані спостереження за квантилями та прагне однакової кількості рядків у кожній групі.

2. **One-hot для міста:** міста номінальні, тому числовий код 1, 2, 3 створив би хибне припущення, що місто 3 «більше» за місто 1. One-hot зберігає окрему ознаку для кожної категорії без штучного порядку.

3. **Масштабування:** методи, чутливі до масштабу, наприклад відстані або регресія з регуляризацією, можуть надати надмірну вагу змінній із великими числовими одиницями. Масштабування робить внесок ознак зіставним.

4. **Похідна змінна:** частка доходу поєднує суму замовлення і фінансові можливості клієнта. Та сама покупка може бути незначною для одного клієнта й істотною для іншого, тому відносна частка часто змістовніша за два вихідні стовпці окремо.

## Підсумок

У наборі клієнтів побудовано похідні змінні, виконано дискретизацію віку через `cut` і `qcut`, нормалізовано вік та дохід двома способами й закодовано місто доставки індикаторами. Кожне перетворення відповідає типу даних і подальшій аналітичній задачі.